<a href="https://colab.research.google.com/github/slxplxt/slxplxt/blob/main/bank-marketing-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 6.9 MB/s eta 0:00:00


In [42]:
import kagglehub

path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bank-marketing-dataset' dataset.
Path to dataset files: /kaggle/input/bank-marketing-dataset


In [59]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

In [44]:
df = pd.read_csv("/kaggle/input/bank-marketing-dataset/bank.csv")

print(df.shape)

(11162, 17)


In [45]:
display(df.head())
print('--------------------')
display(df.tail())
print('--------------------')
display(df.info())
print('--------------------')
print(df.describe())
print('--------------------')
print(df['deposit'].value_counts()['no'])
print('--------------------')
print(df.isna().sum())

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


--------------------


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no
11161,34,technician,married,secondary,no,0,no,no,cellular,9,jul,628,1,-1,0,unknown,no


--------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


None

--------------------
                age       balance           day      duration      campaign  \
count  11162.000000  11162.000000  11162.000000  11162.000000  11162.000000   
mean      41.231948   1528.538524     15.658036    371.993818      2.508421   
std       11.913369   3225.413326      8.420740    347.128386      2.722077   
min       18.000000  -6847.000000      1.000000      2.000000      1.000000   
25%       32.000000    122.000000      8.000000    138.000000      1.000000   
50%       39.000000    550.000000     15.000000    255.000000      2.000000   
75%       49.000000   1708.000000     22.000000    496.000000      3.000000   
max       95.000000  81204.000000     31.000000   3881.000000     63.000000   

              pdays      previous  
count  11162.000000  11162.000000  
mean      51.330407      0.832557  
std      108.758282      2.292007  
min       -1.000000      0.000000  
25%       -1.000000      0.000000  
50%       -1.000000      0.000000  
75%       20.75

1. Так целевая переменная deposit сбалансирована, базовой метрикой будет Accuracy. В качестве основной метрики оценки вероятностей выбран ROC-AUC, так как для банковского бизнеса критически важно, чтобы алгоритм умел корректно ранжировать клиентов.

In [46]:
print(df['education'].value_counts())
print('-----------------------------------------------')
print(df['job'].value_counts())
print('-----------------------------------------------')
print(df['contact'].value_counts())
print('-----------------------------------------------')
print(df['poutcome'].value_counts())
print('-----------------------------------------------')
print(df['pdays'].value_counts())

education
secondary    5476
tertiary     3689
primary      1500
unknown       497
Name: count, dtype: int64
-----------------------------------------------
job
management       2566
blue-collar      1944
technician       1823
admin.           1334
services          923
retired           778
self-employed     405
student           360
unemployed        357
entrepreneur      328
housemaid         274
unknown            70
Name: count, dtype: int64
-----------------------------------------------
contact
cellular     8042
unknown      2346
telephone     774
Name: count, dtype: int64
-----------------------------------------------
poutcome
unknown    8326
failure    1228
success    1071
other       537
Name: count, dtype: int64
-----------------------------------------------
pdays
-1      8324
 92      106
 182      89
 91       84
 181      81
        ... 
 717       1
 159       1
 118       1
 241       1
 15        1
Name: count, Length: 472, dtype: int64


In [47]:
fig_edu = px.histogram(df,
                       x='education',
                       color='deposit',
                       barmode='group',
                       title = 'Распределение открытых депозитов по уровню образования',
                       color_discrete_sequence=["#00CC96", "#EF553B"])
fig_edu.show()

In [48]:
fig_bal = px.box(df, x='deposit', y='balance', color='deposit', title='Распределение баланса в зависимости от решения клиента')
fig_bal.show()

In [49]:
df['contacted_before'] = [1 if x > 0 else 0 for x in df['pdays']]
df['age_group'] = ['students' if x < 25 else 'worker' if x < 61 else 'retiree' for x in df['age']]
display(df)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,contacted_before,age_group
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes,0,worker
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes,0,worker
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes,0,worker
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes,0,worker
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes,0,worker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no,0,worker
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no,0,worker
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no,0,worker
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no,1,worker


2. Изначально была гипотеза, что образование не должно влиять на результат, но после построения графиков она была опровергнута. Также были найдены выбросы, при использовании линейных алгоритмов придется масштабировать данные. Были сгенерированы новые признаки contacted_before и age_group.
При работе с датасетом Bank Marketing эксперты отмечают критическую особенность признака duration (длительность звонка) (https://www.kaggle.com/code/janiobachmann/bank-marketing-campaign-opening-a-term-deposit). Этот признак сильно коррелирует с целевой переменной, но в реальном бизнесе длительность звонка неизвестна до совершения самого звонка. Для построения честной предиктивной модели, работающей на опережение, признак duration рекомендуется исключать из обучения. Однако в рамках построения базового бейзлайна он сохранен.


In [52]:
df['deposit'] = pd.get_dummies(df['deposit'], drop_first=True, dtype = 'int')
X = df.drop(['deposit', 'age', 'pdays'], axis=1)
y = df['deposit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42 )

In [54]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_pred = dummy_model.predict(X_test)
dummy_pred_proba = dummy_model.predict_proba(X_test)[:, 1]

print("--- Константное предсказание ---")
print(f"Accuracy: {accuracy_score(y_test, dummy_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, dummy_pred_proba):.4f}\n")

--- Константное предсказание ---
Accuracy: 0.5222
ROC-AUC: 0.5000



In [56]:
num_cols = ['balance', 'day', 'duration', 'campaign', 'previous', 'contacted_before']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'age_group']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ])

baseline_pipeline = Pipeline(steps =
                             [
                                 ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
                             ])
baseline_pipeline.fit(X_train, y_train)
base_pred = baseline_pipeline.predict(X_test)
base_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]
print("--- Логистическая Регрессия (Бейзлайн) ---")
print(f"Accuracy: {accuracy_score(y_test, base_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, base_pred_proba):.4f}")

--- Логистическая Регрессия (Бейзлайн) ---
Accuracy: 0.8128
ROC-AUC: 0.9038


3. Было проведено измерение качества константного предсказания с помощью DummyClassifier, результаты такие, как будто модель просто угадывала.
Была обучена бейзлайновая модель (Логистическая регрессия), ее показатели гораздо выше константной.

In [60]:
def objective(trial):
  param_grid = {
      'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
  }
  xgb_model = XGBClassifier(**param_grid, random_state=42, eval_metric = 'logloss')

  pipeline = Pipeline(steps=[
      ('preprocessor', preprocessor),
        ('classifier', xgb_model)
  ])
  scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
  return np.mean(scores)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
print("Начинаем подбор гиперпараметров Optuna. Ждем...")
study.optimize(objective, n_trials=15)

print(f"\nЛучший ROC-AUC на кросс-валидации: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}\n")

best_xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_xgb)
])

final_pipeline.fit(X_train, y_train)
final_pred = final_pipeline.predict(X_test)
final_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("--- Финальная Модель: XGBoost (Optuna) ---")
print(f"Accuracy: {accuracy_score(y_test, final_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, final_pred_proba):.4f}")

Начинаем подбор гиперпараметров Optuna. Ждем...

Лучший ROC-AUC на кросс-валидации: 0.9258
Лучшие параметры: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.023778489245362532, 'subsample': 0.6482538746206091}

--- Финальная Модель: XGBoost (Optuna) ---
Accuracy: 0.8442
ROC-AUC: 0.9233


4. Была построена более сложная модель, с подбором параметров с помощью кросс-валидации.

In [67]:
model = final_pipeline.named_steps['classifier']
preprocessor = final_pipeline.named_steps['preprocessor']

num_feature_names = num_cols
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = list(num_feature_names) + list(cat_feature_names)

importances = model.feature_importances_

df_importance = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).head(15)# Берем Топ-15 самых важных

fig = px.bar(df_importance,
             x='Importance',
             y='Feature',
             orientation='h',
             title='Топ-15 самых важных признаков для модели XGBoost',
             color='Importance',
             color_continuous_scale='Viridis')

fig.update_layout(yaxis={'categoryorder':'total ascending'}) # Чтобы самый важный был сверху
fig.show()


Визуализация Feature Importance показывает высокую бизнес-адекватность построенной модели. Наибольший вклад в вероятность открытия депозита вносит признак poutcome_success: успешный опыт прошлых продаж является сильнейшим индикатором лояльности клиента. На втором месте находится contact_unknown, что логично штрафует "холодную" базу контактов. Признак duration (длительность текущего звонка) ожидаемо вошел в Топ-5, подтверждая правило продаж "чем дольше диалог, тем выше шанс сделки". Однако, для использования модели в реальном бизнесе (для предиктивного скоринга базы до начала обзвона) этот признак рекомендуется исключить, так как его значение не может быть известно заранее. Наличие у клиента ипотеки (housing_yes) также справедливо оценивается моделью как негативный фактор, снижающий объем свободного капитала.